# SQL playground: students, courses and scores

A small PostgreSQL database to try SQL on. The first code cell starts an **empty** PostgreSQL server on your computer; the next cells create three small tables and fill them. After that, every cell is one query with comments. Change the queries, break them, and run them again.

The tables:

| Table | One row is | Columns |
|---|---|---|
| `students` | one student | `student_id`, `name`, `programme`, `start_year` |
| `courses` | one course | `course_id`, `title`, `credits` |
| `scores` | one exam result of one student in one course | `student_id`, `course_id`, `exam_date`, `score` |

```mermaid
erDiagram
    students ||--o{ scores : "takes exams"
    courses  ||--o{ scores : "is examined in"
```

To start again from an empty database, restart the kernel and run all cells.

## 1 · Start an empty PostgreSQL server

`pixeltable_pgserver` ships PostgreSQL as a Python package. `get_server(folder)` starts a server whose data live in `folder`. Here the folder is a new temporary folder, and `cleanup_mode="delete"` deletes it when the kernel stops: every start is an empty database.

The server runs **locally**, on this computer only (a socket file on macOS and Linux, `127.0.0.1` on Windows). `server.get_uri("postgres")` returns the address of its default database `postgres`, so you never type a host, port or password.

In [1]:
import tempfile

import pandas as pd
import pixeltable_pgserver as pgserver
import psycopg
from psycopg.types.numeric import FloatLoader

# a new, empty data folder for every run; deleted again when the kernel stops
server = pgserver.get_server(tempfile.mkdtemp(prefix="sql-playground-"), cleanup_mode="delete")

# autocommit=True: every statement is saved at once, and an error does not block the next cell
con = psycopg.connect(server.get_uri("postgres"), autocommit=True)
con.adapters.register_loader("numeric", FloatLoader)   # NUMERIC values arrive as Python floats


def q(sql, params=None):
    """Run one SQL statement. A query returns its rows as a DataFrame; other statements return None."""
    with con.cursor() as cur:
        cur.execute(sql, params)
        if cur.description is None:          # CREATE, INSERT, UPDATE ... return no rows
            return None
        rows = pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])
        return rows.convert_dtypes()         # whole numbers stay whole even with a NULL (shown as <NA>)


q("SELECT version()")

,version
0,PostgreSQL 16.15 on aarch64-apple-darwin24.6.0...


## 2 · Create the three tables

Each column has a name, a type and optional rules (**constraints**):

- `PRIMARY KEY`: identifies a row; unique and never empty.
- `NOT NULL`: the value may not be missing.
- `CHECK (...)`: a condition every row must satisfy.
- `REFERENCES other_table (column)`: a **foreign key**; the value must exist in the other table.

`DROP TABLE IF EXISTS ... CASCADE` removes old copies first, so you can run this cell again.

In [2]:
q("""
DROP TABLE IF EXISTS scores, courses, students CASCADE;   -- start clean; scores first, it refers to the others

CREATE TABLE students (
    student_id  INTEGER PRIMARY KEY,                     -- one row per student
    name        TEXT NOT NULL,
    programme   TEXT NOT NULL,                           -- e.g. 'Data Science'
    start_year  INTEGER CHECK (start_year >= 2000)       -- NULL allowed: unknown
);

CREATE TABLE courses (
    course_id   TEXT PRIMARY KEY,                        -- a short code such as 'SQL'
    title       TEXT NOT NULL,
    credits     INTEGER NOT NULL CHECK (credits > 0)
);

CREATE TABLE scores (
    student_id  INTEGER NOT NULL REFERENCES students (student_id),   -- foreign key to students
    course_id   TEXT    NOT NULL REFERENCES courses (course_id),     -- foreign key to courses
    exam_date   DATE    NOT NULL,
    score       NUMERIC(5, 1) CHECK (score BETWEEN 0 AND 100),       -- NULL allowed: exam not graded yet
    PRIMARY KEY (student_id, course_id)                              -- one result per student and course
);
""")

## 3 · Fill the tables with sample rows

`INSERT INTO table (columns) VALUES (...), (...)` adds several rows at once. `NULL` marks a missing value: Lena's start year is unknown, and one exam is not graded yet. Kai has no exam at all, which the joins below will show.

In [3]:
q("""
INSERT INTO students (student_id, name, programme, start_year) VALUES
    (1, 'Amira',  'Data Science',     2025),
    (2, 'Ben',    'Data Science',     2025),
    (3, 'Chen',   'Business Analytics', 2024),
    (4, 'Dilara', 'Business Analytics', 2025),
    (5, 'Emil',   'Data Science',     2024),
    (6, 'Lena',   'Economics',        NULL),
    (7, 'Kai',    'Economics',        2025);

INSERT INTO courses (course_id, title, credits) VALUES
    ('SQL',  'Relational databases and SQL', 5),
    ('STAT', 'Statistics',                    6),
    ('ML',   'Machine learning',              6);

INSERT INTO scores (student_id, course_id, exam_date, score) VALUES
    (1, 'SQL',  '2026-01-20', 92),
    (1, 'STAT', '2026-02-03', 78),
    (1, 'ML',   '2026-02-10', 85),
    (2, 'SQL',  '2026-01-20', 64),
    (2, 'STAT', '2026-02-03', 55),
    (3, 'SQL',  '2026-01-20', 88),
    (3, 'ML',   '2026-02-10', 91),
    (4, 'SQL',  '2026-01-20', 71),
    (4, 'STAT', '2026-02-03', 82),
    (4, 'ML',   '2026-02-10', NULL),       -- not graded yet
    (5, 'STAT', '2026-02-03', 49),
    (5, 'ML',   '2026-02-10', 67),
    (6, 'SQL',  '2026-01-20', 95);
""")

# count the rows of each table
q("""
SELECT (SELECT COUNT(*) FROM students) AS students,
       (SELECT COUNT(*) FROM courses)  AS courses,
       (SELECT COUNT(*) FROM scores)   AS scores
""")

,students,courses,scores
0,7,3,13


## 4 · Look at the tables

`SELECT *` returns every column. `information_schema.columns` describes the columns of every table.

In [4]:
q("SELECT * FROM students")

,student_id,name,programme,start_year
0,1,Amira,Data Science,2025
1,2,Ben,Data Science,2025
2,3,Chen,Business Analytics,2024
3,4,Dilara,Business Analytics,2025
4,5,Emil,Data Science,2024
5,6,Lena,Economics,<NA>
6,7,Kai,Economics,2025


In [5]:
# the columns of the table scores, with their types
q("""
SELECT column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_name = 'scores'
ORDER BY ordinal_position
""")

,column_name,data_type,is_nullable
0,student_id,integer,NO
1,course_id,text,NO
2,exam_date,date,NO
3,score,numeric,YES


## 5 · Filter and sort: `WHERE`, `ORDER BY`, `LIMIT`

In [6]:
# all SQL exams with a score of at least 80, best first
q("""
SELECT student_id, score
FROM scores
WHERE course_id = 'SQL' AND score >= 80       -- both conditions must hold
ORDER BY score DESC                           -- DESC: largest first
""")

,student_id,score
0,6,95
1,1,92
2,3,88


In [7]:
# text patterns: ILIKE ignores upper and lower case, % stands for any text
q("""
SELECT name, programme
FROM students
WHERE programme ILIKE '%data%'
ORDER BY name
LIMIT 3                                       -- only the first three rows
""")

,name,programme
0,Amira,Data Science
1,Ben,Data Science
2,Emil,Data Science


## 6 · Missing values: `NULL`

A comparison with `NULL` is never true, so `score = NULL` finds nothing. Use `IS NULL`. `COUNT(*)` counts rows, `COUNT(score)` only the rows with a score.

In [8]:
q("""
SELECT COUNT(*)      AS all_results,
       COUNT(score)  AS graded,                -- skips the NULL score
       (SELECT COUNT(*) FROM scores WHERE score = NULL)  AS equals_null,    -- always 0
       (SELECT COUNT(*) FROM scores WHERE score IS NULL) AS is_null
FROM scores
""")

,all_results,graded,equals_null,is_null
0,13,12,0,1


## 7 · Combine tables: `JOIN`

`JOIN ... ON` puts the rows of two tables side by side where the condition holds. Short **table aliases** (`s`, `c`, `st`) say which table a column comes from.

In [9]:
# every result with the student's name and the course title
q("""
SELECT st.name, c.title, s.score
FROM scores AS s
JOIN students AS st ON st.student_id = s.student_id   -- inner join: only matching rows
JOIN courses  AS c  ON c.course_id  = s.course_id
ORDER BY st.name, c.title
""")

,name,title,score
0,Amira,Machine learning,85
1,Amira,Relational databases and SQL,92
2,Amira,Statistics,78
3,Ben,Relational databases and SQL,64
4,Ben,Statistics,55
5,Chen,Machine learning,91
6,Chen,Relational databases and SQL,88
7,Dilara,Machine learning,<NA>
8,Dilara,Relational databases and SQL,71
9,Dilara,Statistics,82


In [10]:
# LEFT JOIN keeps every student, also those without any exam (their columns from scores are NULL)
q("""
SELECT st.name, s.course_id, s.score
FROM students AS st
LEFT JOIN scores AS s ON s.student_id = st.student_id
WHERE s.student_id IS NULL                    -- keep only students without a result
""")

,name,course_id,score
0,Kai,None,None


## 8 · Summarise: `GROUP BY` and `HAVING`

`GROUP BY` makes one row per group; aggregate functions (`COUNT`, `AVG`, `MIN`, `MAX`, `SUM`) summarise each group. `WHERE` filters rows before grouping, `HAVING` filters groups after.

In [11]:
# number of exams and average score per course
q("""
SELECT course_id,
       COUNT(*)             AS n_exams,
       ROUND(AVG(score), 1) AS avg_score,       -- AVG skips the NULL score
       MAX(score)           AS best
FROM scores
GROUP BY course_id
ORDER BY avg_score DESC
""")

,course_id,n_exams,avg_score,best
0,SQL,5,82,95
1,ML,4,81,91
2,STAT,4,66,82


In [12]:
# students whose average is above 75, with their credits passed (score of 50 or more)
q("""
SELECT st.name,
       ROUND(AVG(s.score), 1)                         AS avg_score,
       SUM(c.credits) FILTER (WHERE s.score >= 50)     AS credits_passed
FROM scores AS s
JOIN students AS st ON st.student_id = s.student_id
JOIN courses  AS c  ON c.course_id  = s.course_id
GROUP BY st.name
HAVING AVG(s.score) > 75                       -- a condition on the group
ORDER BY avg_score DESC
""")

,name,avg_score,credits_passed
0,Lena,95.0,5
1,Chen,89.5,11
2,Amira,85.0,17
3,Dilara,76.5,11


## 9 · Name an intermediate result: `WITH` (a common table expression)

A **CTE** gives a query result a name; the main query uses it like a table, as often as it needs.

In [13]:
# students who scored above the average of their course
q("""
WITH course_avg AS (                           -- step 1: one average per course
    SELECT course_id, AVG(score) AS avg_score
    FROM scores
    GROUP BY course_id
)
SELECT st.name, s.course_id, s.score, ROUND(ca.avg_score, 1) AS course_average
FROM scores AS s
JOIN course_avg AS ca ON ca.course_id = s.course_id   -- step 2: compare each score with it
JOIN students   AS st ON st.student_id = s.student_id
WHERE s.score > ca.avg_score
ORDER BY s.course_id, s.score DESC
""")

,name,course_id,score,course_average
0,Chen,ML,91,81
1,Amira,ML,85,81
2,Lena,SQL,95,82
3,Amira,SQL,92,82
4,Chen,SQL,88,82
5,Dilara,STAT,82,66
6,Amira,STAT,78,66


## 10 · Rank within groups: window functions

A **window function** computes a value for every row from a group of rows (the window), without merging the rows. `RANK() OVER (PARTITION BY course_id ORDER BY score DESC)` numbers the results within each course, best first.

In [14]:
q("""
SELECT course_id, student_id, score,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC NULLS LAST) AS rank_in_course
FROM scores
ORDER BY course_id, rank_in_course
""")

,course_id,student_id,score,rank_in_course
0,ML,3,91,1
1,ML,1,85,2
2,ML,5,67,3
3,ML,4,<NA>,4
4,SQL,6,95,1
5,SQL,1,92,2
6,SQL,3,88,3
7,SQL,4,71,4
8,SQL,2,64,5
9,STAT,4,82,1


## 11 · Change rows: `UPDATE` and `DELETE`

`RETURNING` shows the rows a statement changed. Without `WHERE`, `UPDATE` and `DELETE` act on **every** row.

In [15]:
# Dilara's ML exam is graded now
q("UPDATE scores SET score = 74 WHERE student_id = 4 AND course_id = 'ML' RETURNING *")

,student_id,course_id,exam_date,score
0,4,ML,2026-02-10,74


In [16]:
# remove one result again
q("DELETE FROM scores WHERE student_id = 2 AND course_id = 'STAT' RETURNING *")

,student_id,course_id,exam_date,score
0,2,STAT,2026-02-03,55


## 12 · The rules in action

Each statement below breaks one constraint. The database refuses it and the message names the rule. `try`/`except` prints the message, so the notebook keeps running.

In [17]:
broken = {
    "unknown student (foreign key)": "INSERT INTO scores VALUES (99, 'SQL', '2026-01-20', 70)",
    "same student and course twice": "INSERT INTO scores VALUES (1, 'SQL', '2026-03-01', 60)",
    "score above 100 (CHECK)":       "INSERT INTO scores VALUES (7, 'SQL', '2026-01-20', 120)",
    "not a date":                    "INSERT INTO scores VALUES (7, 'SQL', '2026-02-30', 70)",
}
for label, statement in broken.items():
    try:
        q(statement)
        print(f"{label:32} accepted")
    except psycopg.Error as error:
        print(f"{label:32} rejected: {error.diag.message_primary}")

unknown student (foreign key)    rejected: insert or update on table "scores" violates foreign key constraint "scores_student_id_fkey"
same student and course twice    rejected: duplicate key value violates unique constraint "scores_pkey"
score above 100 (CHECK)          rejected: new row for relation "scores" violates check constraint "scores_score_check"
not a date                       rejected: date/time field value out of range: "2026-02-30"


## 13 · Your turn

Some ideas:

1. Add yourself as a student and one exam result.
2. Which programme has the highest average score?
3. Which course has no result from an Economics student?
4. Add a fourth table, for example `lecturers`, and a column `lecturer_id` in `courses` that refers to it (`ALTER TABLE courses ADD COLUMN ...`).

## 14 · Stop the server

Closing the connection and calling `server.cleanup()` stops PostgreSQL and deletes the temporary data folder. Restarting the kernel does the same.

In [18]:
con.close()
server.cleanup()          # stops the server and deletes the playground data